# 31_global-plus-residual — Shared global head with linear residual experts

Independent three-seed expert-parameterization ablation against Notebook 31. The encoder and linear `64 → 4` gate remain unchanged. Each expert instead predicts `global_head(z) + residual_e(z)`: the shared global head is initialized from the exact pooled Stage-A CE probe, every `64 → 22` residual starts at zero, and Stage B trains only the owned residual. Because this expert bank differs from Notebook 31, Stage A is reused but Stage B and Stage C are retrained. The ordinary all-trainable Notebook 31 Stage-C schedule is retained so this notebook does not import the separate gate-only freezing intervention.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('baseline_independent_experts', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('global_plus_residual', 'config/compact_soft_moe_global_residual_3seed.yaml', 'nfv3_4way_moe_global_residual_v1'),
]
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS_PER_STUDY = 3
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight, dry-run, and resumable execution

Only the signed Stage-A checkpoint is reused. The challenger must independently create its global-residual Stage-B bank before Stage C, preventing a baseline expert checkpoint from being loaded into the new parameterization.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
residual = yaml.safe_load(Path('config/compact_soft_moe_global_residual_3seed.yaml').read_text())
assert residual['reuse_stage_a_from_prefix'] == 'nfv3_4way_moe_soft_comparable_repro_v1'
assert 'reuse_stage_b_from_prefix' not in residual
assert residual['backbone']['architecture'] == 'moe_dataset_global_residual'
assert residual['backbone']['model']['encoder']['hidden_dims'] == [128]
assert residual['backbone']['model']['expert']['hidden_dims'] == []
assert residual['backbone']['model']['gate']['hidden_dims'] == []
assert residual['backbone']['training']['stage_c_unfreeze'] == 'all'
assert not residual['backbone']['training']['stage_c'].get('freeze_experts', False)
assert residual['expected_total_parameters'] == 21810
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_global_residual_experts.py', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    print(f'=== {condition}: architecture={study["backbone"]["architecture"]}, parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-global-residual-{condition}-dry-run')
    if EXECUTE: run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-global-residual-{condition}-training')
if not EXECUTE: print('NO TRAINING WAS STARTED. Review the contracts, set EXECUTE=True, and rerun this cell.')

## Paired validation, locked-test, per-class, and resource comparison

In [ ]:
import pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}; print('Completion:', complete)
if all(complete.values()):
    validation = []; test = []; classes = []; resources = []
    for name, path in summaries.items():
        for filename, target in [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Three_Seed_Resource_Accounting.csv', resources)]:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True); test = pd.concat(test, ignore_index=True)
    display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1')
    pivot['delta_global_residual_minus_baseline'] = pivot['global_plus_residual'] - pivot['baseline_independent_experts']; display(pivot)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    display(pd.concat(resources, ignore_index=True))
else: print('Comparison appears after both three-seed manifests are complete.')

## Global initialization and residual diagnostics

The Stage-B global head must remain exactly equal to the Stage-A pooled CE probe because Stage B optimizes only residual parameters. Residual norms show how much specialization each dataset learned; Stage-C global drift is reported separately because the ordinary all-trainable schedule is intentionally retained.

In [ ]:
if complete.get('global_plus_residual'):
    import numpy as np
    prefix = 'nfv3_4way_moe_global_residual_v1'; baseline = 'nfv3_4way_moe_soft_comparable_repro_v1'
    rows = []
    for seed in (0, 1, 2):
        stage_a = torch.load(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{baseline}_seed{seed}' / 'stage_a_encoder.pt', map_location='cpu')
        stage_b = torch.load(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{prefix}_seed{seed}' / 'stage_b_expert_bank.pt', map_location='cpu')['expert_bank_state']
        stage_c = torch.load(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{prefix}_seed{seed}' / 'stage_c_full.pt', map_location='cpu')['model_state']
        a_weight = stage_a['representation_state']['classifier.linear.weight']; a_bias = stage_a['representation_state']['classifier.linear.bias']
        assert torch.equal(stage_b['global_head.weight'], a_weight) and torch.equal(stage_b['global_head.bias'], a_bias)
        global_drift = float(torch.cat([(stage_c['expert_bank.global_head.weight'] - a_weight).flatten(), stage_c['expert_bank.global_head.bias'] - a_bias]).norm())
        for expert_id, expert_name in enumerate(['NF-UNSW-NB15-v3', 'NF-ToN-IoT-v3', 'NF-BoT-IoT-v3', 'NF-CICIDS2018-v3']):
            b_weight = stage_b[f'residuals.{expert_id}.weight']; b_bias = stage_b[f'residuals.{expert_id}.bias']
            c_weight = stage_c[f'expert_bank.residuals.{expert_id}.weight']; c_bias = stage_c[f'expert_bank.residuals.{expert_id}.bias']
            rows.append({'seed': seed, 'expert': expert_name, 'stage_b_residual_l2': float(torch.cat([b_weight.flatten(), b_bias]).norm()), 'stage_c_residual_l2': float(torch.cat([c_weight.flatten(), c_bias]).norm()), 'stage_c_global_drift_l2': global_drift})
    display(pd.DataFrame(rows))
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        print(f'=== Seed {seed}: routing and utilization ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv')); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))
        print(f'=== Seed {seed}: owned experts after Stage C ==='); display(pd.read_csv(result_dir / 'Validation_Expert_Owned_StageC.csv'))

## A/B/C training and latent diagnostics

In [ ]:
if all(complete.values()):
    for name, path in summaries.items():
        print('===', name, 'training history ===')
        history = pd.read_csv(path / 'Training_History_3Seed.csv')
        display(history[['seed', 'stage', 'dataset', 'epoch', 'learning_rate', 'train_total_loss', 'val_macro_f1', 'best_val_macro_f1']])
        display(pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'))
        for figure in sorted((path / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))